> **About this notebook**
>
> This notebook is part of the example gallery of [CLABTOOLKIT](https://github.com/connectomicslab/clabtoolkit), an open-source Python toolkit for routine processing, manipulation and visualization of neuroimaging data. It shows how to use the `morphometrytools` module.
>
> 📦 **Install:** `pip install clabtoolkit` ([PyPI](https://pypi.org/project/clabtoolkit/)) · 📖 **Docs:** [clabtoolkit.readthedocs.io](https://clabtoolkit.readthedocs.io) · 📄 **Paper:** [arXiv:2607.02638](https://arxiv.org/abs/2607.02638)
>
> **Author:** Yasser Alemán-Gómez (Connectomics Lab, Radiology Department, CHUV – Lausanne University Hospital)

## Testing Methods in the `MORPHOMETRYTOOLS` Module

This Jupyter Notebook serves as a platform to rigorously test the various methods included in the Python module `MORPHOMETRYTOOLS`.

This module is particularly valuable for researchers extracting morphometric measurements from
neuroimaging data: computing regional statistics from surface metrics and annotations, surface area,
number of vertices and Euler characteristics from meshes, regional statistics and volumes from
volumetric parcellations, parsing FreeSurfer `aseg.stats` and `aparc.stats` files, and summarizing
connectivity matrices into network metric tables.

The tests are organized into the following sections for clarity and ease of navigation:

### Table of Contents

---

* 🔷 1. [`Metrics from Surfaces`](#morpho-surfaces). Methods dedicated to compute metrics from surfaces and surface meshes
    * 🔸 1.1. [`compute_reg_val_fromannot`](#morpho-compute_reg_val_fromannot). Compute regional statistics from a surface metric file and an annotation file
    * 🔸 1.2. [`compute_reg_area_fromsurf`](#morpho-compute_reg_area_fromsurf). Compute the surface area for each region defined in an annotation file
    * 🔸 1.3. [`compute_reg_nvertices_fromsurf`](#morpho-compute_reg_nvertices_fromsurf). Compute the number of vertices for each region defined in an annotation file
    * 🔸 1.4. [`compute_euler_fromsurf`](#morpho-compute_euler_fromsurf). Compute the Euler characteristic of a surface mesh
    * 🔸 1.5. [`area_from_mesh`](#morpho-area_from_mesh). Compute the total area and the per-triangle areas of a mesh surface
    * 🔸 1.6. [`euler_from_mesh`](#morpho-euler_from_mesh). Compute the Euler characteristic of a mesh surface

* 🔷 2. [`Metrics from Parcellations`](#morpho-parcellations). Methods dedicated to compute metrics from volumetric parcellations
    * 🔸 2.1. [`compute_reg_val_fromparcellation`](#morpho-compute_reg_val_fromparcellation). Compute regional statistics from a volumetric metric map and a parcellation
    * 🔸 2.2. [`compute_reg_volume_fromparcellation`](#morpho-compute_reg_volume_fromparcellation). Compute the volume for all the regions in a parcellation

* 🔷 3. [`Parsing FreeSurfer Stats Files`](#morpho-freesurfer_stats). Methods dedicated to parse stats files from FreeSurfer results
    * 🔸 3.1. [`parse_freesurfer_global_fromaseg`](#morpho-parse_freesurfer_global_fromaseg). Parse global measurements (volumes, areas and unitless measures) from a FreeSurfer aseg.stats file
    * 🔸 3.2. [`parse_freesurfer_stats_fromaseg`](#morpho-parse_freesurfer_stats_fromaseg). Parse regional volume measurements from a FreeSurfer aseg.stats file
    * 🔸 3.3. [`parse_freesurfer_cortex_stats`](#morpho-parse_freesurfer_cortex_stats). Parse cortical parcellation statistics from a FreeSurfer aparc.stats file
    * 🔸 3.4. [`get_stats_dictionary`](#morpho-get_stats_dictionary). Return the default measurement configuration (global, aseg or cortex) for FreeSurfer stats files

* 🔷 4. [`Metrics from Connectivity`](#morpho-connectivity). Methods dedicated to extract metrics from connectivity matrices
    * 🔸 4.1. [`network_metrics_to_table`](#morpho-network_metrics_to_table). Compute network metrics from a connectivity matrix and return them in a DataFrame

* 🔷 5. [`Auxiliary Methods`](#morpho-auxiliary). Auxiliary methods used to summarize metrics and handle their units
    * 🔸 5.1. [`stats_from_vector`](#morpho-stats_from_vector). Compute the specified statistics from a numeric vector
    * 🔸 5.2. [`get_units`](#morpho-get_units). Get the units associated with the specified metrics
    * 🔸 5.3. [`convert_from_mm`](#morpho-convert_from_mm). Convert areas in mm² or volumes in mm³ to the units defined in config.json

---

---
## <a id="morpho-surfaces"></a>1. Metrics from Surfaces
Methods dedicated to compute metrics from surfaces and surface meshes.

### 1.1 `compute_reg_val_fromannot`
<a id="morpho-compute_reg_val_fromannot"></a>

Compute regional statistics from a surface metric file and an annotation file.

In [ ]:
######################## Testing compute_reg_val_fromannot ########################
import os
import clabtoolkit.morphometrytools as cltmorpho
import clabtoolkit.freesurfertools as cltfree
import itables
import clabtoolkit.surfacetools as cltsurf

freesurfer_home = os.environ.get("FREESURFER_HOME")
if freesurfer_home is None:

    ######################### First Surface (Left Hemisphere) ###########################
    # Create Surface object 
    surf_obj_lh  = cltsurf.Surface.simulate_surface(seed=42, shape="sphere") 
    surf_obj_lh.save_surface(filename="/tmp/simulated_surface_lh.surf", overwrite=True)
    ct_map_file_lh = "/tmp/simulated_surface_lh.thickness"
    curv_map_file_lh = "/tmp/simulated_surface_lh.curv"
    sulc_map_file_lh = "/tmp/simulated_surface_lh.sulc"
    surf_file_lh = "/tmp/simulated_surface_lh.surf"
    annot_file_lh = "/tmp/simulated_surface_lh.annot"

    # Simulating annotation file
    annot_obj = cltfree.AnnotParcellation.simulate_annotation(surf_obj_lh, n_regions=10)
    annot_obj.save_annotation(annot_file_lh, overwrite=True)

    # Simulating thickness file
    surf_obj_lh.simulate_map(map_name="Thickness")
    surf_obj_lh.export_map(filename=ct_map_file_lh, overlay_name="Thickness", overwrite=True)

    # Simulating curvature file
    surf_obj_lh.simulate_map(map_name="Curvature")
    surf_obj_lh.export_map(filename=curv_map_file_lh, overlay_name="Curvature", overwrite=True)

    # Simulating sulcal depth file
    surf_obj_lh.simulate_map(map_name="Sulc")
    surf_obj_lh.export_map(filename=sulc_map_file_lh, overlay_name="Sulc", overwrite=True)

    ######################### Second Surface (Right Hemisphere) ###########################
    # Create Surface object for the right hemisphere
    surf_obj_rh  = cltsurf.Surface.simulate_surface(seed=42, shape="torus") 
    surf_obj_rh.save_surface(filename="/tmp/simulated_surface_rh.surf", overwrite=True)
    ct_map_file_rh = "/tmp/simulated_surface_rh.thickness"
    curv_map_file_rh = "/tmp/simulated_surface_rh.curv"
    sulc_map_file_rh = "/tmp/simulated_surface_rh.sulc"
    surf_file_rh = "/tmp/simulated_surface_rh.surf"
    annot_file_rh = "/tmp/simulated_surface_rh.annot"

    # Simulating annotation file
    annot_obj = cltfree.AnnotParcellation.simulate_annotation(surf_obj_rh, n_regions=10)
    annot_obj.save_annotation(annot_file_rh, overwrite=True)

    # Simulating thickness file
    surf_obj_rh.simulate_map(map_name="Thickness")
    surf_obj_rh.export_map(filename=ct_map_file_rh, overlay_name="Thickness", overwrite=True)

    # Simulating curvature file
    surf_obj_rh.simulate_map(map_name="Curvature")
    surf_obj_rh.export_map(filename=curv_map_file_rh, overlay_name="Curvature", overwrite=True)

    # Simulating sulcal depth file
    surf_obj_rh.simulate_map(map_name="Sulc")
    surf_obj_rh.export_map(filename=sulc_map_file_rh, overlay_name="Sulc", overwrite=True)

    
else:
    subjects_dir = os.path.join(freesurfer_home, 'subjects')
    subject_id = 'fsaverage'
    surf_id = 'pial'
    hemi_id = 'lh'
    surf_file_lh = os.path.join(subjects_dir, subject_id, 'surf', f'{hemi_id}.{surf_id}')
    annot_file_lh = os.path.join(subjects_dir, subject_id, 'label', f'{hemi_id}.aparc.annot')
    ct_map_file_lh = os.path.join(subjects_dir, subject_id, 'surf', f'{hemi_id}.thickness')
    curv_map_file_lh = os.path.join(subjects_dir, subject_id, 'surf', f'{hemi_id}.curv')
    sulc_map_file_lh = os.path.join(subjects_dir, subject_id, 'surf', f'{hemi_id}.sulc')

    hemi_id = 'rh'
    annot_file_rh = os.path.join(subjects_dir, subject_id, 'label', f'{hemi_id}.aparc.annot')
    surf_file_rh = os.path.join(subjects_dir, subject_id, 'surf', f'{hemi_id}.{surf_id}')
    ct_map_file_rh = os.path.join(subjects_dir, subject_id, 'surf', f'{hemi_id}.thickness')
    curv_map_file_rh = os.path.join(subjects_dir, subject_id, 'surf', f'{hemi_id}.curv')
    sulc_map_file_rh = os.path.join(subjects_dir, subject_id, 'surf', f'{hemi_id}.sulc')

maps_names = ["Thickness", "Curvature", "Sulc"]


## Left Hemisphere
#  - Load surface
surf_lh = cltsurf.Surface(surf_file_lh)

#  - Load scalar maps
surf_lh.load_scalar_maps(ct_map_file_lh, maps_names='thickness')
surf_lh.load_scalar_maps(curv_map_file_lh, maps_names="curvature")
surf_lh.load_scalar_maps(sulc_map_file_lh, maps_names="sulc")
surf_lh.load_annotation(annot_file_lh, parc_name='aparc')

## Right Hemisphere
#  - Load surface

surf_rh = cltsurf.Surface(surf_file_rh)
#  - Load scalar maps
surf_rh.load_scalar_maps(ct_map_file_rh, maps_names='thickness')
surf_rh.load_scalar_maps(curv_map_file_rh, maps_names="curvature")
surf_rh.load_scalar_maps(sulc_map_file_rh, maps_names="sulc")
surf_rh.load_annotation(annot_file_rh, parc_name='aparc')

# Test 1: Basic usage with metric format
print('\nTest 1: Basic usage with metric format')
hemi = "lh"
metric_name = "Thickness"
print(f"Input: metric_file={ct_map_file_lh}, parc_file={annot_file_lh}, hemi={hemi}, metric={metric_name}, include_global=False")
df_region, metric_values, _ = cltmorpho.compute_reg_val_fromannot(
    ct_map_file_lh, annot_file_lh, hemi, metric=metric_name, include_global=False
)
itables.show(df_region)

### 1.2 `compute_reg_area_fromsurf`
<a id="morpho-compute_reg_area_fromsurf"></a>

Compute the surface area for each region defined in an annotation file.

In [ ]:
######################## Testing compute_reg_area_fromsurf ########################
import os
import numpy as np
import clabtoolkit.morphometrytools as cltmorpho
import clabtoolkit.freesurfertools as cltfree
import itables
import clabtoolkit.surfacetools as cltsurf

freesurfer_home = os.environ.get("FREESURFER_HOME")
if freesurfer_home is None:

    ######################### First Surface (Left Hemisphere) ###########################
    # Create Surface object 
    surf_obj_lh  = cltsurf.Surface.simulate_surface(seed=42, shape="sphere") 
    surf_obj_lh.save_surface(filename="/tmp/simulated_surface_lh.surf", overwrite=True)
    ct_map_file_lh = "/tmp/simulated_surface_lh.thickness"
    curv_map_file_lh = "/tmp/simulated_surface_lh.curv"
    sulc_map_file_lh = "/tmp/simulated_surface_lh.sulc"
    surf_file_lh = "/tmp/simulated_surface_lh.surf"
    annot_file_lh = "/tmp/simulated_surface_lh.annot"

    # Simulating annotation file
    annot_obj = cltfree.AnnotParcellation.simulate_annotation(surf_obj_lh, n_regions=10)
    annot_obj.save_annotation(annot_file_lh, overwrite=True)

    # Simulating thickness file
    surf_obj_lh.simulate_map(map_name="Thickness")
    surf_obj_lh.export_map(filename=ct_map_file_lh, overlay_name="Thickness", overwrite=True)

    # Simulating curvature file
    surf_obj_lh.simulate_map(map_name="Curvature")
    surf_obj_lh.export_map(filename=curv_map_file_lh, overlay_name="Curvature", overwrite=True)

    # Simulating sulcal depth file
    surf_obj_lh.simulate_map(map_name="Sulc")
    surf_obj_lh.export_map(filename=sulc_map_file_lh, overlay_name="Sulc", overwrite=True)

    ######################### Second Surface (Right Hemisphere) ###########################
    # Create Surface object for the right hemisphere
    surf_obj_rh  = cltsurf.Surface.simulate_surface(seed=42, shape="torus") 
    surf_obj_rh.save_surface(filename="/tmp/simulated_surface_rh.surf", overwrite=True)
    ct_map_file_rh = "/tmp/simulated_surface_rh.thickness"
    curv_map_file_rh = "/tmp/simulated_surface_rh.curv"
    sulc_map_file_rh = "/tmp/simulated_surface_rh.sulc"
    surf_file_rh = "/tmp/simulated_surface_rh.surf"
    annot_file_rh = "/tmp/simulated_surface_rh.annot"

    # Simulating annotation file
    annot_obj = cltfree.AnnotParcellation.simulate_annotation(surf_obj_rh, n_regions=10)
    annot_obj.save_annotation(annot_file_rh, overwrite=True)

    # Simulating thickness file
    surf_obj_rh.simulate_map(map_name="Thickness")
    surf_obj_rh.export_map(filename=ct_map_file_rh, overlay_name="Thickness", overwrite=True)

    # Simulating curvature file
    surf_obj_rh.simulate_map(map_name="Curvature")
    surf_obj_rh.export_map(filename=curv_map_file_rh, overlay_name="Curvature", overwrite=True)

    # Simulating sulcal depth file
    surf_obj_rh.simulate_map(map_name="Sulc")
    surf_obj_rh.export_map(filename=sulc_map_file_rh, overlay_name="Sulc", overwrite=True)

    surf_type = "simulated"
else:
    subjects_dir = os.path.join(freesurfer_home, 'subjects')
    subject_id = 'fsaverage'
    surf_id = 'pial'
    hemi_id = 'lh'
    surf_file_lh = os.path.join(subjects_dir, subject_id, 'surf', f'{hemi_id}.{surf_id}')
    annot_file_lh = os.path.join(subjects_dir, subject_id, 'label', f'{hemi_id}.aparc.annot')
    ct_map_file_lh = os.path.join(subjects_dir, subject_id, 'surf', f'{hemi_id}.thickness')
    curv_map_file_lh = os.path.join(subjects_dir, subject_id, 'surf', f'{hemi_id}.curv')
    sulc_map_file_lh = os.path.join(subjects_dir, subject_id, 'surf', f'{hemi_id}.sulc')

    hemi_id = 'rh'
    annot_file_rh = os.path.join(subjects_dir, subject_id, 'label', f'{hemi_id}.aparc.annot')
    surf_file_rh = os.path.join(subjects_dir, subject_id, 'surf', f'{hemi_id}.{surf_id}')
    ct_map_file_rh = os.path.join(subjects_dir, subject_id, 'surf', f'{hemi_id}.thickness')
    curv_map_file_rh = os.path.join(subjects_dir, subject_id, 'surf', f'{hemi_id}.curv')
    sulc_map_file_rh = os.path.join(subjects_dir, subject_id, 'surf', f'{hemi_id}.sulc')
    surf_type = surf_id

maps_names = ["Thickness", "Curvature", "Sulc"]


## Left Hemisphere
#  - Load surface
surf_lh = cltsurf.Surface(surf_file_lh)

#  - Load scalar maps
surf_lh.load_scalar_maps(ct_map_file_lh, maps_names='thickness')
surf_lh.load_scalar_maps(curv_map_file_lh, maps_names="curvature")
surf_lh.load_scalar_maps(sulc_map_file_lh, maps_names="sulc")
surf_lh.load_annotation(annot_file_lh, parc_name='aparc')

## Right Hemisphere
#  - Load surface

surf_rh = cltsurf.Surface(surf_file_rh)
#  - Load scalar maps
surf_rh.load_scalar_maps(ct_map_file_rh, maps_names='thickness')
surf_rh.load_scalar_maps(curv_map_file_rh, maps_names="curvature")
surf_rh.load_scalar_maps(sulc_map_file_rh, maps_names="sulc")
surf_rh.load_annotation(annot_file_rh, parc_name='aparc')

# Set the hemisphere for the tests
hemi = 'lh'

# Test 1: Basic usage with metric format
print('\nTest 1: Basic usage with metric format')
print(f"Input: surf_file={surf_file_lh}, parc_file={annot_file_lh}, hemi={hemi}, surf_type={surf_type}")
df_area, _ = cltmorpho.compute_reg_area_fromsurf(
    surf_file_lh, annot_file_lh, hemi, surf_type=surf_type
)
itables.show(df_area)

# Test 2: Using region format
print('\nTest 2: Using region format for output')
print(f"Input: surf_file={surf_file_lh}, parc_file={annot_file_lh}, hemi={hemi}, table_type='region', surf_type={surf_type}")
df_region, _ = cltmorpho.compute_reg_area_fromsurf(
    surf_file_lh, annot_file_lh, hemi, 
    table_type="region", surf_type=surf_type
)
itables.show(df_region)

# Test 3: Without global statistics
print('\nTest 3: Excluding hemisphere-wide statistics')
print(f"Input: surf_file={surf_file_lh}, parc_file={annot_file_lh}, hemi={hemi}, include_global=False, surf_type={surf_type}")
df_no_global, _ = cltmorpho.compute_reg_area_fromsurf(
    surf_file_lh, annot_file_lh, hemi, include_global=False, surf_type=surf_type
)
print(f"Result shape: {df_no_global.shape}")
print(f"Contains global statistics: {'ctx-lh-hemisphere' in df_no_global['Region'].values}")
itables.show(df_no_global)

# Test 4: Saving output to a file
print('\nTest 4: Saving output to a TSV file')
output_path = '/tmp/area_stats.tsv'
print(f"Input: surf_file={surf_file_lh}, parc_file={annot_file_lh}, hemi={hemi}, output_table={output_path}, surf_type={surf_type}")
df_output, table_path = cltmorpho.compute_reg_area_fromsurf(
    surf_file_lh, annot_file_lh, hemi, output_table=output_path, 
    surf_type=surf_type
)
print(f"Output table path: {table_path}")
print(f"File exists: {os.path.exists(table_path)}")
print(f"File size: {os.path.getsize(table_path)} bytes")

# Test 5: Using Surface and AnnotParcellation objects
print('\nTest 5: Using objects instead of file paths')
surf = cltsurf.Surface(surface_file=surf_file_lh)
annot = cltfree.AnnotParcellation(parc_file=annot_file_lh)
print(f"Input: surf_file=<Surface object>, parc_file=<AnnotParcellation object>, hemi={hemi}, surf_type={surf_type}")
df_objects, _ = cltmorpho.compute_reg_area_fromsurf(
    surf, annot, hemi, surf_type=surf_type
)
print(f"Result shape: {df_objects.shape}")
# The object input has no file name, so the BIDS and MetricFile columns are absent:
# compare the values only
print(f"Values match file-based approach: {np.allclose(df_objects['Value'], df_area['Value'])}")

# Test 6: Including unknown regions
print('\nTest 6: Including unknown regions')
print(f"Input: surf_file={surf_file_lh}, parc_file={annot_file_lh}, hemi={hemi}, include_unknown=True, surf_type={surf_type}")
df_with_unknown, _ = cltmorpho.compute_reg_area_fromsurf(
    surf_file_lh, annot_file_lh, hemi, include_unknown=True, surf_type=surf_type
)
print(f"Result shape: {df_with_unknown.shape}")
print(f"Row count comparison: {df_with_unknown.shape[0]} vs {df_area.shape[0]}")

# Test 7: Without BIDS entities
print('\nTest 7: Without BIDS entities')
print(f"Input: surf_file={surf_file_lh}, parc_file={annot_file_lh}, hemi={hemi}, add_bids_entities=False, surf_type={surf_type}")
df_no_bids, _ = cltmorpho.compute_reg_area_fromsurf(
    surf_file_lh, annot_file_lh, hemi, add_bids_entities=False, surf_type=surf_type
)
print(f"Result shape: {df_no_bids.shape}")
print(f"Column count comparison: {df_no_bids.shape[1]} (without BIDS) vs {df_area.shape[1]} (with BIDS)")

# Test 8: Error handling for non-existent files
print('\nTest 8: Error handling for non-existent files')
nonexistent_file = '/path/to/nonexistent/file.surf'
print(f"Input: surf_file={nonexistent_file}, parc_file={annot_file_lh}, hemi={hemi}")
try:
    df_error, _ = cltmorpho.compute_reg_area_fromsurf(
        nonexistent_file, annot_file_lh, hemi
    )
    print("No error raised (unexpected)")
except FileNotFoundError as e:
    print(f"Expected error raised: {type(e).__name__}: {str(e)}")

# Summary of test results
print('\nSummary of compute_reg_area_fromsurf Tests')
print('-------------------------------------------')
print('✓ Test 1: Basic usage with metric format')
print('✓ Test 2: Using region format for output')
print('✓ Test 3: Excluding hemisphere-wide statistics')
print('✓ Test 4: Saving output to a TSV file')
print('✓ Test 5: Using objects instead of file paths')
print('✓ Test 6: Including unknown regions')
print('✓ Test 7: Without BIDS entities')
print('✓ Test 8: Error handling for non-existent files')


### 1.3 `compute_reg_nvertices_fromsurf`
<a id="morpho-compute_reg_nvertices_fromsurf"></a>

Compute the number of vertices for each region defined in an annotation file.

In [ ]:
######################## Testing compute_reg_nvertices_fromsurf ########################
import os
import clabtoolkit.morphometrytools as cltmorpho
import clabtoolkit.freesurfertools as cltfree
import itables
import clabtoolkit.surfacetools as cltsurf

freesurfer_home = os.environ.get("FREESURFER_HOME")
if freesurfer_home is None:

    ######################### First Surface (Left Hemisphere) ###########################
    # Create Surface object 
    surf_obj_lh  = cltsurf.Surface.simulate_surface(seed=42, shape="sphere") 
    surf_obj_lh.save_surface(filename="/tmp/simulated_surface_lh.surf", overwrite=True)
    surf_file_lh = "/tmp/simulated_surface_lh.surf"
    annot_file_lh = "/tmp/simulated_surface_lh.annot"

    # Simulating annotation file
    annot_obj = cltfree.AnnotParcellation.simulate_annotation(surf_obj_lh, n_regions=10)
    annot_obj.save_annotation(annot_file_lh, overwrite=True)

    ######################### Second Surface (Right Hemisphere) ###########################
    # Create Surface object for the right hemisphere
    surf_obj_rh  = cltsurf.Surface.simulate_surface(seed=42, shape="torus") 
    surf_obj_rh.save_surface(filename="/tmp/simulated_surface_rh.surf", overwrite=True)
    surf_file_rh = "/tmp/simulated_surface_rh.surf"
    annot_file_rh = "/tmp/simulated_surface_rh.annot"

    # Simulating annotation file
    annot_obj = cltfree.AnnotParcellation.simulate_annotation(surf_obj_rh, n_regions=10)
    annot_obj.save_annotation(annot_file_rh, overwrite=True)

    surf_type = "simulated"
else:
    subjects_dir = os.path.join(freesurfer_home, 'subjects')
    subject_id = 'fsaverage'
    surf_id = 'pial'
    hemi_id = 'lh'
    surf_file_lh = os.path.join(subjects_dir, subject_id, 'surf', f'{hemi_id}.{surf_id}')
    annot_file_lh = os.path.join(subjects_dir, subject_id, 'label', f'{hemi_id}.aparc.annot')

    hemi_id = 'rh'
    annot_file_rh = os.path.join(subjects_dir, subject_id, 'label', f'{hemi_id}.aparc.annot')
    surf_file_rh = os.path.join(subjects_dir, subject_id, 'surf', f'{hemi_id}.{surf_id}')
    surf_type = surf_id

hemi = 'lh'

# Test 1: Basic usage with metric format
print('\nTest 1: Basic usage with metric format')
print(f"Input: surf_file={surf_file_lh}, parc_file={annot_file_lh}, hemi=lh, surf_type={surf_type}")
df_nvertices, _ = cltmorpho.compute_reg_nvertices_fromsurf(
    surf_file_lh, annot_file_lh, hemi=hemi, surf_type=surf_type
)
itables.show(df_nvertices)

hemi = 'rh'
print('\nTest 2: Right hemisphere without global statistics')
print(f"Input: surf_file={surf_file_rh}, parc_file={annot_file_rh}, hemi=rh, surf_type={surf_type}, include_global=False")
df_nvertices, _ = cltmorpho.compute_reg_nvertices_fromsurf(
    surf_file_rh, annot_file_rh, hemi=hemi, surf_type=surf_type, include_global=False
)
itables.show(df_nvertices)  

print('\nTest 3: Basic usage with Surface and AnnotParcellation objects for left hemisphere')
print(f"Input: surf_file=<Surface object>, parc_file=<AnnotParcellation object>, hemi=lh, surf_type={surf_type}")
## Left Hemisphere
#  - Load surface
surf_lh = cltsurf.Surface(surf_file_lh)
annot_lh = cltfree.AnnotParcellation(annot_file_lh)

hemi = 'lh'
df_nvertices_lh, _ = cltmorpho.compute_reg_nvertices_fromsurf(
    surf_lh, annot_lh, hemi=hemi, surf_type=surf_type
)
itables.show(df_nvertices_lh)   

### 1.4 `compute_euler_fromsurf`
<a id="morpho-compute_euler_fromsurf"></a>

Compute the Euler characteristic of a surface mesh.

In [ ]:
######################## Testing compute_euler_fromsurf ########################
import os
import clabtoolkit.morphometrytools as cltmorpho
import itables
import clabtoolkit.surfacetools as cltsurf

freesurfer_home = os.environ.get("FREESURFER_HOME")
if freesurfer_home is None:

    ######################### First Surface (Left Hemisphere) ###########################
    # Create Surface object 
    surf_obj_lh  = cltsurf.Surface.simulate_surface(seed=42, shape="sphere") 
    surf_obj_lh.save_surface(filename="/tmp/simulated_surface_lh.surf", overwrite=True)
    surf_file_lh = "/tmp/simulated_surface_lh.surf"

    surf_type = "simulated"
else:
    subjects_dir = os.path.join(freesurfer_home, 'subjects')
    subject_id = 'fsaverage'
    surf_id = 'pial'
    hemi_id = 'lh'
    surf_file_lh = os.path.join(subjects_dir, subject_id, 'surf', f'{hemi_id}.{surf_id}')
    surf_type = surf_id

hemi = 'lh'

# Test 1: Basic usage with metric format
print('\nTest 1: Basic usage with metric format')
print(f"Input: surf_file={surf_file_lh}, hemi=lh, surf_type={surf_type}")
df_euler, _ = cltmorpho.compute_euler_fromsurf(
    surf_file_lh, hemi=hemi, surf_type=surf_type
)
itables.show(df_euler)

# Test 2: Using a Surface object for the left hemisphere
print('\nTest 2: Using a Surface object for the left hemisphere')
hemi = 'lh'
print(f"Input: surf_file={surf_file_lh}, hemi=lh, surf_type={surf_type}")
surf_obj_lh = cltsurf.Surface(surf_file_lh)
df_euler, _ = cltmorpho.compute_euler_fromsurf(
    surf_obj_lh, hemi=hemi, surf_type=surf_type
)
itables.show(df_euler)

### 1.5 `area_from_mesh`
<a id="morpho-area_from_mesh"></a>

Compute the total area and the per-triangle areas of a mesh surface.

In [ ]:
######################## Testing area_from_mesh ########################
import os
import clabtoolkit.morphometrytools as cltmorpho
import clabtoolkit.surfacetools as cltsurf

freesurfer_home = os.environ.get("FREESURFER_HOME")
if freesurfer_home is None:

    ######################### First Surface (Left Hemisphere) ###########################
    # Create Surface object 
    surf_obj_lh  = cltsurf.Surface.simulate_surface(seed=42, shape="sphere") 
    surf_obj_lh.save_surface(filename="/tmp/simulated_surface_lh.surf", overwrite=True)
    surf_file_lh = "/tmp/simulated_surface_lh.surf"

    surf_type = "simulated"
else:
    subjects_dir = os.path.join(freesurfer_home, 'subjects')
    subject_id = 'fsaverage'
    surf_id = 'pial'
    hemi_id = 'lh'
    surf_file_lh = os.path.join(subjects_dir, subject_id, 'surf', f'{hemi_id}.{surf_id}')
    surf_type = surf_id

# Test 1: Basic usage with metric format
print('\nTest 1: Basic usage with metric format')
print(f"Input: surf_file={surf_file_lh}, surf_type={surf_type}")

surf_obj_lh = cltsurf.Surface(surf_file_lh)
coords = surf_obj_lh.get_vertices()
faces = surf_obj_lh.get_faces()
total_area, tri_area = cltmorpho.area_from_mesh(
    coords, faces
)
area_units = cltmorpho.get_units("area")[0]
print(f"Output: total area = {total_area:.2f} {area_units} ({tri_area.size} triangles)")

### 1.6 `euler_from_mesh`
<a id="morpho-euler_from_mesh"></a>

Compute the Euler characteristic of a mesh surface.

In [ ]:
######################## Testing euler_from_mesh ########################
import os
import clabtoolkit.morphometrytools as cltmorpho
import clabtoolkit.surfacetools as cltsurf

freesurfer_home = os.environ.get("FREESURFER_HOME")
if freesurfer_home is None:

    ######################### First Surface (Left Hemisphere) ###########################
    # Create Surface object 
    surf_obj_lh  = cltsurf.Surface.simulate_surface(seed=42, shape="sphere") 
    surf_obj_lh.save_surface(filename="/tmp/simulated_surface_lh.surf", overwrite=True)
    surf_file_lh = "/tmp/simulated_surface_lh.surf"

    surf_type = "simulated"
else:
    subjects_dir = os.path.join(freesurfer_home, 'subjects')
    subject_id = 'fsaverage'
    surf_id = 'pial'
    hemi_id = 'lh'
    surf_file_lh = os.path.join(subjects_dir, subject_id, 'surf', f'{hemi_id}.{surf_id}')
    surf_type = surf_id

# Test 1: Basic usage with metric format
print('\nTest 1: Basic usage with metric format')
print(f"Input: surf_file={surf_file_lh}, surf_type={surf_type}")

surf_obj_lh = cltsurf.Surface(surf_file_lh)
coords = surf_obj_lh.get_vertices()
faces = surf_obj_lh.get_faces()
euler = cltmorpho.euler_from_mesh(
    coords, faces
)
print(f"Output: Euler characteristic = {euler}")

---
## <a id="morpho-parcellations"></a>2. Metrics from Parcellations
Methods dedicated to compute metrics from volumetric parcellations.

### 2.1 `compute_reg_val_fromparcellation`
<a id="morpho-compute_reg_val_fromparcellation"></a>

Compute regional statistics from a volumetric metric map and a parcellation.

In [ ]:
######################## Testing compute_reg_val_fromparcellation ########################
import clabtoolkit.parcellationtools as cltparc
import clabtoolkit.morphometrytools as cltmorpho
import clabtoolkit.imagetools as cltimg
import os
import itables

freesurfer_home = os.environ.get("FREESURFER_HOME")
if freesurfer_home is None:
    parc = cltparc.Parcellation.simulate_parcellation(n_regions=10, seed=42)
    parc.set_names(['ctx-rh-precentral-simu', 'ctx-lh-precentral-simu', 'ctx-rh-postcentral-simu',
                    'ctx-lh-postcentral-simu', 'ctx-rh-superiorfrontal-simu', 'ctx-lh-superiorfrontal-simu',
                    'ctx-rh-inferiorparietal-simu', 'ctx-lh-inferiorparietal-simu',
                    'ctx-rh-supramarginal-simu', 'ctx-lh-supramarginal-simu'])
    t1_file = "/tmp/simu_T1.nii.gz"
    cltimg.simulate_image(parc, simulated_image= t1_file)
    st_codes = [1, 2]  # codes of ctx-rh-precentral-simu and ctx-lh-precentral-simu
    st_names = ['ctx-rh-precentral-simu', 'ctx-lh-precentral-simu']

else:
    subjects_dir = os.path.join(freesurfer_home, 'subjects')
    subject_id = 'bert'
    parc_id = 'aparc+aseg.mgz'
    t1 = 'T1.mgz'
    parc_file = os.path.join(subjects_dir, subject_id, 'mri', f'{parc_id}')
    lut_file = os.path.join(freesurfer_home, 'FreeSurferColorLUT.txt')
    t1_file = os.path.join(subjects_dir, subject_id, 'mri', f'{t1}')
    parc = cltparc.Parcellation(parc_file, color_table=lut_file)

    st_codes = [10, 49] # Thalamus codes according to FreeSurferColorLUT.txt
    st_names = ['Thalamus'] # Thalamus names according to FreeSurferColorLUT.txt

# Test 1: Compute regional values from parcellation
print(f"\nTest 1: Compute regional values from parcellation")
df_reg_val, _, _ = cltmorpho.compute_reg_val_fromparcellation(
    t1_file, parc, metric="intensity"
)
itables.show(df_reg_val)

# Test 2: Compute regional values from parcellation for specific structures
print(f"\nTest 2: Compute regional values from parcellation for specific structures")
df_reg_val_specific, _, _ = cltmorpho.compute_reg_val_fromparcellation(
    t1_file, parc, metric="intensity", include_by_name=st_names
)
itables.show(df_reg_val_specific)

# Test 3: Compute regional values from parcellation excluding specific structures
print(f"\nTest 3: Compute regional values from parcellation excluding specific structures")
df_reg_val_exclude, _, _ = cltmorpho.compute_reg_val_fromparcellation(
    t1_file, parc, metric="intensity", exclude_by_name=st_names
)
itables.show(df_reg_val_exclude)

# Test 4: Compute regional values from parcellation including regions with certain codes
print(f"\nTest 4: Compute regional values from parcellation including regions with certain codes")
df_reg_val_include_codes, _, _ = cltmorpho.compute_reg_val_fromparcellation(
    t1_file, parc, metric="intensity", include_by_code=st_codes
)
itables.show(df_reg_val_include_codes)

# Test 5: Compute regional values from parcellation excluding regions with certain codes
print(f"\nTest 5: Compute regional values from parcellation excluding regions with certain codes")
df_reg_val_exclude_codes, _, _ = cltmorpho.compute_reg_val_fromparcellation(
    t1_file, parc, metric="intensity", exclude_by_code=st_codes
)
itables.show(df_reg_val_exclude_codes)

### 2.2 `compute_reg_volume_fromparcellation`
<a id="morpho-compute_reg_volume_fromparcellation"></a>

Compute the volume for all the regions in a parcellation.

In [ ]:
######################## Testing compute_reg_volume_fromparcellation ########################
import clabtoolkit.parcellationtools as cltparc
import clabtoolkit.morphometrytools as cltmorpho
import clabtoolkit.imagetools as cltimg
import os
import itables

freesurfer_home = os.environ.get("FREESURFER_HOME")
if freesurfer_home is None:
    parc = cltparc.Parcellation.simulate_parcellation(n_regions=10, seed=42)
    parc.set_names(['ctx-rh-precentral-simu', 'ctx-lh-precentral-simu', 'ctx-rh-postcentral-simu',
                    'ctx-lh-postcentral-simu', 'ctx-rh-superiorfrontal-simu', 'ctx-lh-superiorfrontal-simu',
                    'ctx-rh-inferiorparietal-simu', 'ctx-lh-inferiorparietal-simu',
                    'ctx-rh-supramarginal-simu', 'ctx-lh-supramarginal-simu'])
    t1_file = "/tmp/simu_T1.nii.gz"
    cltimg.simulate_image(parc, simulated_image= t1_file)
    st_codes = [1, 2]  # codes of ctx-rh-precentral-simu and ctx-lh-precentral-simu
    st_names = ['ctx-rh-precentral-simu', 'ctx-lh-precentral-simu']

else:
    subjects_dir = os.path.join(freesurfer_home, 'subjects')
    subject_id = 'bert'
    parc_id = 'aparc+aseg.mgz'
    t1 = 'T1.mgz'
    parc_file = os.path.join(subjects_dir, subject_id, 'mri', f'{parc_id}')
    lut_file = os.path.join(freesurfer_home, 'FreeSurferColorLUT.txt')
    t1_file = os.path.join(subjects_dir, subject_id, 'mri', f'{t1}')
    parc = cltparc.Parcellation(parc_file, color_table=lut_file)

    st_codes = [10, 49] # Thalamus codes according to FreeSurferColorLUT.txt
    st_names = ['Thalamus'] # Thalamus names according to FreeSurferColorLUT.txt

# Test 1: Compute regional volumes from parcellation
print(f"\nTest 1: Compute regional volumes from parcellation")
df_reg_vol, _ = cltmorpho.compute_reg_volume_fromparcellation(
    parc
)
itables.show(df_reg_vol)

# Test 2: Compute regional volumes from parcellation for specific structures
print(f"\nTest 2: Compute regional volumes from parcellation for specific structures")
df_reg_vol_specific, _ = cltmorpho.compute_reg_volume_fromparcellation(
    parc, include_by_name=st_names
)
itables.show(df_reg_vol_specific)

# Test 3: Compute regional volumes from parcellation excluding specific structures
print(f"\nTest 3: Compute regional volumes from parcellation excluding specific structures")
df_reg_vol_exclude, _ = cltmorpho.compute_reg_volume_fromparcellation(
    parc, exclude_by_name=st_names
)
itables.show(df_reg_vol_exclude)

# Test 4: Compute regional volumes from parcellation including regions with certain codes
print(f"\nTest 4: Compute regional volumes from parcellation including regions with certain codes")
df_reg_vol_include_codes, _ = cltmorpho.compute_reg_volume_fromparcellation(
    parc, include_by_code=st_codes
)
itables.show(df_reg_vol_include_codes)

# Test 5: Compute regional volumes from parcellation excluding regions with certain codes
print(f"\nTest 5: Compute regional volumes from parcellation excluding regions with certain codes")
df_reg_vol_exclude_codes, _ = cltmorpho.compute_reg_volume_fromparcellation(
    parc, exclude_by_code=st_codes
)
itables.show(df_reg_vol_exclude_codes)

# Test 6: Excluding the global value
print(f"\nTest 6: Excluding the global value")
df_reg_vol_exclude_global, _ = cltmorpho.compute_reg_volume_fromparcellation(
    parc, include_global=False
)
itables.show(df_reg_vol_exclude_global)

---
## <a id="morpho-freesurfer_stats"></a>3. Parsing FreeSurfer Stats Files
Methods dedicated to parse stats files from FreeSurfer results.

### 3.1 `parse_freesurfer_global_fromaseg`
<a id="morpho-parse_freesurfer_global_fromaseg"></a>

Parse global measurements (volumes, areas and unitless measures) from a FreeSurfer aseg.stats file.

In [ ]:
######################## Testing parse_freesurfer_global_fromaseg ########################
import clabtoolkit.morphometrytools as cltmorpho
import os
import itables

freesurfer_home = os.environ.get("FREESURFER_HOME")
if freesurfer_home is not None:

    subjects_dir = os.path.join(freesurfer_home, 'subjects')
    subject_id = 'bert'
    aseg_name= 'aseg.stats'
    aseg_file = os.path.join(subjects_dir, subject_id, 'stats', f'{aseg_name}')

    if os.path.exists(aseg_file):
        print(f"Found aseg file: {aseg_file}")
    else:
        raise FileNotFoundError(f"Aseg file not found: {aseg_file}")

    aseg_global_table,_ = cltmorpho.parse_freesurfer_global_fromaseg(aseg_file)
    itables.show(aseg_global_table)
else:
    print("FREESURFER_HOME is not set: skipping this example "
          "(it needs the FreeSurfer 'bert' subject).")


### 3.2 `parse_freesurfer_stats_fromaseg`
<a id="morpho-parse_freesurfer_stats_fromaseg"></a>

Parse regional volume measurements from a FreeSurfer aseg.stats file.

In [ ]:
######################## Testing parse_freesurfer_stats_fromaseg ########################
import clabtoolkit.morphometrytools as cltmorpho
import os
import itables

freesurfer_home = os.environ.get("FREESURFER_HOME")
if freesurfer_home is not None:

    subjects_dir = os.path.join(freesurfer_home, 'subjects')
    subject_id = 'bert'
    aseg_name= 'aseg.stats'
    aseg_file = os.path.join(subjects_dir, subject_id, 'stats', f'{aseg_name}')

    if os.path.exists(aseg_file):
        print(f"Found aseg file: {aseg_file}")
    else:
        raise FileNotFoundError(f"Aseg file not found: {aseg_file}")

    aseg_table,_ = cltmorpho.parse_freesurfer_stats_fromaseg(aseg_file)
    itables.show(aseg_table)
else:
    print("FREESURFER_HOME is not set: skipping this example "
          "(it needs the FreeSurfer 'bert' subject).")


### 3.3 `parse_freesurfer_cortex_stats`
<a id="morpho-parse_freesurfer_cortex_stats"></a>

Parse cortical parcellation statistics from a FreeSurfer aparc.stats file.

In [ ]:
######################## Testing parse_freesurfer_cortex_stats ########################
import clabtoolkit.morphometrytools as cltmorpho
import os
import itables

freesurfer_home = os.environ.get("FREESURFER_HOME")
if freesurfer_home is not None:

    subjects_dir = os.path.join(freesurfer_home, 'subjects')
    subject_id = 'bert'
    aparc_name= 'lh.aparc.stats'
    aparc_file = os.path.join(subjects_dir, subject_id, 'stats', f'{aparc_name}')

    if os.path.exists(aparc_file):
        print(f"Found aparc file: {aparc_file}")
    else:
        raise FileNotFoundError(f"Aparc file not found: {aparc_file}")

    aparc_table,_ = cltmorpho.parse_freesurfer_cortex_stats(aparc_file)
    itables.show(aparc_table)
else:
    print("FREESURFER_HOME is not set: skipping this example "
          "(it needs the FreeSurfer 'bert' subject).")


### 3.4 `get_stats_dictionary`
<a id="morpho-get_stats_dictionary"></a>

Return the default measurement configuration (global, aseg or cortex) for FreeSurfer stats files.

In [ ]:
######################## Testing get_stats_dictionary ########################
import clabtoolkit.morphometrytools as cltmorpho
import clabtoolkit.misctools as cltmisc

# Test 1: Get global information dictionary
print("\nTest 1: Testing get_stats_dictionary with 'global'")
global_reg = cltmorpho.get_stats_dictionary("global")
tmp = cltmisc.ExplorerDict(global_reg)
tmp.tree()

# Test 2: Get regional information dictionary
print("\nTest 2: Testing get_stats_dictionary with 'aseg'")
aseg_reg = cltmorpho.get_stats_dictionary("aseg")
tmp = cltmisc.ExplorerDict(aseg_reg)
tmp.tree()

# Test 3: Get regional information dictionary for 'cortex'
print("\nTest 3: Testing get_stats_dictionary with 'cortex'")
cortex_reg = cltmorpho.get_stats_dictionary("cortex")
tmp = cltmisc.ExplorerDict(cortex_reg)
tmp.tree()


---
## <a id="morpho-connectivity"></a>4. Metrics from Connectivity
Methods dedicated to extract metrics from connectivity matrices.

### 4.1 `network_metrics_to_table`
<a id="morpho-network_metrics_to_table"></a>

Compute network metrics from a connectivity matrix and return them in a DataFrame.

In [ ]:
######################## Testing network_metrics_to_table ###################################
import clabtoolkit.connectivitytools as cltconn
import clabtoolkit.morphometrytools as cltmorpho
import itables
import os
import tempfile

import numpy as np

# Test 1: Generate metrics table from a random Connectome object
print('Test 1: Generate a random connectome')
conn_random = cltconn.Connectome.generate_connectome(20, 
                                method='random', seed=0, sparsity=0.20)
conn_random.plot_matrix(figsize=(5, 4))
print('')

conn_table,_ = cltmorpho.network_metrics_to_table(conn_random)

itables.show(conn_table)

# Test 2: Generate metrics table from a random symmetric numpy matrix
print("\n Test 2: Generate metrics table from a random symmetric numpy matrix")
rng = np.random.default_rng(0)
conn_random_np = rng.random((20, 20))
conn_random_np = (conn_random_np + conn_random_np.T) / 2  # Make it symmetric
np.fill_diagonal(conn_random_np, 0)  # No self-connections
# The matrix is fully connected: all degrees are equal, so the binary assortativity
# is undefined (NaN, with a RuntimeWarning from bctpy). This is expected.
conn_table_np,_ = cltmorpho.network_metrics_to_table(conn_random_np)
itables.show(conn_table_np) 

# Test 3: Select a single metric (string)
print("\n Test 3: Compute a single metric (string)")
deg_table, _ = cltmorpho.network_metrics_to_table(conn_random, metrics="degree")
itables.show(deg_table)

# Test 4: Select several metrics (list)
print("\n Test 4: Compute several metrics (list)")
sel_table, _ = cltmorpho.network_metrics_to_table(
    conn_random,
    cmat_met="nstreamlines",
    metrics=["degree", "strength", "glob_efficiency", "modularity"],
    seed=0,
)
itables.show(sel_table)

# Test 5: Region names from a LUT dictionary, "region" table layout, and saving to TSV
print("\n Test 5: LUT names, region table type and saving the table")
n_nodes = conn_random_np.shape[0]
lut_dict = {
    "index": list(range(1, n_nodes + 1)),
    "name": [f"ctx-lh-region{i:02d}" for i in range(1, n_nodes + 1)],
}

out_file = os.path.join(tempfile.mkdtemp(), "network_metrics.tsv")
reg_table, out_path = cltmorpho.network_metrics_to_table(
    conn_random_np,
    lut_file=lut_dict,
    metrics=["degree", "strength", "clustering_coeff_wei", "transitivity_wei"],
    table_type="region",
    output_table=out_file,
)
print(f"Table saved to: {out_path}")
itables.show(reg_table)

# Test 6: Nodal metrics only, reproducible communities, and an invalid metric name
print("\n Test 6: include_global=False, seed reproducibility and error handling")
comm_metrics = ["participation_coeff", "within_module_zscore", "modularity"]

t1, _ = cltmorpho.network_metrics_to_table(
    conn_random, metrics=comm_metrics, include_global=False, seed=42
)
t2, _ = cltmorpho.network_metrics_to_table(
    conn_random, metrics=comm_metrics, include_global=False, seed=42
)
print("Same seed gives identical results:", t1["Value"].equals(t2["Value"]))
print("Metrics computed:", t1["Metric"].unique().tolist())  # modularity skipped
itables.show(t1)

try:
    cltmorpho.network_metrics_to_table(conn_random, metrics=["degree", "pagerank"])
except ValueError as err:
    print(f"Expected error: {err}")

# Test 7: Save a 'metric' layout table to a file
print("\n Test 7: Save a \'metric\' layout table to a file")
out_file = os.path.join(tempfile.mkdtemp(), "network_metrics.tsv")
saved_table, saved_path = cltmorpho.network_metrics_to_table(
    conn_random_np,
    metrics=["degree", "strength", "clustering_coeff_wei", "transitivity_wei"],
    output_table=out_file,
)
print(f"Table saved to: {saved_path}")
itables.show(saved_table)



---
## <a id="morpho-auxiliary"></a>5. Auxiliary Methods
Auxiliary methods used to summarize metrics and handle their units.

### 5.1 `stats_from_vector`
<a id="morpho-stats_from_vector"></a>

Compute the specified statistics from a numeric vector.

In [ ]:
######################## Testing stats_from_vector ###################################
import numpy as np
import clabtoolkit.morphometrytools as cltmorpho

vals = np.array([0, 1, 2, 3, 4, 0])

# Test 1: Default statistics ["value", "median", "std", "min", "max"]
# Note: nonzeros_only=True by default, so the zeros are ignored
print("Test 1: Default statistics (zeros ignored)")
print(cltmorpho.stats_from_vector(vals))
# [2.5, 2.5, 1.118..., 1.0, 4.0]

# Test 2: Same vector keeping the zeros
print("\n Test 2: Keeping the zeros")
print(cltmorpho.stats_from_vector(vals, nonzeros_only=False))
# [1.666..., 1.5, 1.490..., 0.0, 4.0]

# Test 3: Choosing statistics (case-insensitive, any order)
print("\n Test 3: Custom statistics")
stats = ["MAX", "Mean"]
print(dict(zip(stats, cltmorpho.stats_from_vector(vals, stats, nonzeros_only=False))))
# {'MAX': 4.0, 'Mean': 1.666...}

# Test 4: Any array-like input is flattened (lists, 2D arrays)
print("\n Test 4: List and 2D array inputs")
print(cltmorpho.stats_from_vector([5, 10, 15], ["mean", "count"]))
print(cltmorpho.stats_from_vector(np.arange(1, 7).reshape(2, 3), ["mean", "count"]))
# [10.0, 3.0]
# [3.5, 6.0]

# Test 5: Edge cases: empty and all-zero inputs
print("\n Test 5: Empty and all-zero inputs")
stats = ["mean", "std", "count", "sum"]
print("Empty:   ", cltmorpho.stats_from_vector(np.array([]), stats))
print("All-zero:", cltmorpho.stats_from_vector(np.zeros(5), stats))
# Empty:    [nan, nan, 0.0, 0.0]
# All-zero: [0.0, 0.0, 0.0, 0.0]

# Test 6: NaN values propagate to the statistics
print("\n Test 6: NaN in the input")
print(cltmorpho.stats_from_vector(np.array([1.0, np.nan, 3.0]), ["mean", "max", "count"]))
# [nan, nan, 3.0]

# Test 7: Invalid inputs
print("\n Test 7: Error handling")
for bad_stats in (["mean", "mode"], "mean"):
    try:
        cltmorpho.stats_from_vector(vals, bad_stats)
    except (ValueError, TypeError) as err:
        print(f"{type(err).__name__}: {err}")
# ValueError: Unsupported statistics: mode
# TypeError: stats_list must be a list or tuple

### 5.2 `get_units`
<a id="morpho-get_units"></a>

Get the units associated with the specified metrics.

In [ ]:
######################## Testing get_units ###################################
import json
import os
import tempfile

import clabtoolkit.morphometrytools as cltmorpho

# Test 1: Single metric with the default config.json
print("Test 1: Single metric")
print(cltmorpho.get_units("thickness"))
# ['mm']   (always a list, even for a single metric)

# Test 2: Several metrics, case-insensitive
print("\n Test 2: Several metrics, mixed case")
print(cltmorpho.get_units(["thickness", "Area", "VOLUME"]))
# ['mm', 'mm2', 'mm3']   (with the default config.json)

# Test 3: Metrics missing from the configuration
print("\n Test 3: Unknown metrics")
print(cltmorpho.get_units(["thickness", "my_new_metric"]))
# ['mm', 'unknown']

# Test 4: Custom units from a dictionary (with or without the "metrics_units" key)
print("\n Test 4: Units from a dictionary")
custom = {"metrics_units": {"FA": "au", "MD": "mm2/s"}}
print(cltmorpho.get_units(["fa", "md"], metrics_json=custom))
print(cltmorpho.get_units(["fa", "md"], metrics_json={"FA": "au", "MD": "mm2/s"}))
# ['au', 'mm2/s']
# ['au', 'mm2/s']

# Test 5: Custom units from a JSON file
print("\n Test 5: Units from a JSON file")
tmp_dir = tempfile.mkdtemp()
json_file = os.path.join(tmp_dir, "my_units.json")
with open(json_file, "w", encoding="utf-8") as f:
    json.dump({"metrics_units": {"thickness": "um", "myelin": "au"}}, f)

print(cltmorpho.get_units(["thickness", "myelin", "area"], metrics_json=json_file))
# ['um', 'au', 'unknown']   (only the file is used, not merged with config.json)

# Test 6: Error handling
print("\n Test 6: Error handling")
bad_json = os.path.join(tmp_dir, "bad.json")
with open(bad_json, "w", encoding="utf-8") as f:
    f.write("{not valid json")

no_key_json = os.path.join(tmp_dir, "no_key.json")
with open(no_key_json, "w", encoding="utf-8") as f:
    json.dump({"thickness": "mm"}, f)

for source in ("does_not_exist.json", bad_json, no_key_json, 42):
    try:
        cltmorpho.get_units("thickness", metrics_json=source)
    except ValueError as err:
        print(f"ValueError: {err}")
# ValueError: Invalid JSON file path: does_not_exist.json
# ValueError: Invalid JSON format in file: .../bad.json
# ValueError: Missing 'metrics_units' key in the provided JSON file
# ValueError: metrics_json must be a file path, dictionary, or None

### 5.3 `convert_from_mm`
<a id="morpho-convert_from_mm"></a>

Convert areas in mm² or volumes in mm³ to the units defined in config.json.

In [ ]:
######################## Testing convert_from_mm ###################################
import numpy as np
import clabtoolkit.morphometrytools as cltmorpho

# Test 1: Scalars converted to cm (explicit units, independent of config.json)
print("Test 1: Scalar conversion")
cm_units = {"area": "cm2", "volume": "cm3"}
print(cltmorpho.convert_from_mm(250.0, "area", cm_units))
print(cltmorpho.convert_from_mm(1500.0, "volume", cm_units))
# (2.5, 'cm2')
# (1.5, 'cm3')

# Test 2: Arrays and lists (returned as float64 arrays), case-insensitive metric
print("\n Test 2: Array and list inputs")
print(cltmorpho.convert_from_mm(np.array([100, 250, 1000]), "Area", cm_units))
print(cltmorpho.convert_from_mm([1000, 2500], "VOLUME", cm_units))
# (array([ 1. ,  2.5, 10. ]), 'cm2')
# (array([1. , 2.5]), 'cm3')

# Test 3: Other length units through a custom configuration
print("\n Test 3: Custom target units")
for units, metric, value in [
    ({"area": "mm2"}, "area", 250.0),       # no conversion
    ({"area": "m2"}, "area", 1e6),          # 1 m² = 1e6 mm²
    ({"area": "um2"}, "area", 1.0),         # 1 mm² = 1e6 µm²
    ({"volume": "dm3"}, "volume", 1e6),     # 1 dm³ = 1e6 mm³
]:
    print(units, "->", cltmorpho.convert_from_mm(value, metric, metrics_json=units))
# {'area': 'mm2'} -> (250.0, 'mm2')
# {'area': 'm2'} -> (1.0, 'm2')
# {'area': 'um2'} -> (1000000.0, 'um2')
# {'volume': 'dm3'} -> (1.0, 'dm3')

# Test 4: Volume aliases and unit notations
print("\n Test 4: Aliases (ml, cc, l) and notations (^, ², ³)")
for unit in ["ml", "cc", "l", "cm^3", "cm³", "mm^3"]:
    print(f"{unit:>5}:", cltmorpho.convert_from_mm(2000.0, "volume", {"volume": unit}))
# ml: (2.0, 'ml')     cc: (2.0, 'cc')     l: (0.002, 'l')
# cm^3: (2.0, 'cm^3') cm³: (2.0, 'cm³')   mm^3: (2000.0, 'mm^3')

# Test 5: Consistency with area_from_mesh (a 10 x 10 mm square, both in config units)
print("\n Test 5: Consistency with area_from_mesh")
coords = np.array([[0, 0, 0], [10, 0, 0], [0, 10, 0], [10, 10, 0]])
faces = np.array([[0, 1, 2], [1, 3, 2]])
total_area, _ = cltmorpho.area_from_mesh(coords, faces)
print(total_area, "==", cltmorpho.convert_from_mm(100.0, "area")[0])
# 100.0 == 100.0 with "area": "mm2" in config.json (1.0 == 1.0 with "cm2")

# Test 6: Error handling
print("\n Test 6: Error handling")
cases = [
    (100.0, "thickness", None),               # only area and volume are supported
    (100.0, "volume", {"area": "cm2"}),       # no unit defined for volume
    (100.0, "area", {"area": "cm3"}),         # wrong dimension
    (100.0, "area", {"area": "in2"}),         # unsupported length unit
]
for value, metric, units in cases:
    try:
        cltmorpho.convert_from_mm(value, metric, metrics_json=units)
    except ValueError as err:
        print(f"ValueError: {err}")
# ValueError: convert_from_mm only handles 'area' and 'volume', got 'thickness'.
# ValueError: No unit defined for 'volume' in the units configuration.
# ValueError: Unsupported unit 'cm3' for a quantity of dimension 2.
# ValueError: Unsupported unit 'in2' for a quantity of dimension 2.